In [4]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("twitter_transformation").getOrCreate()

df = spark.read.parquet("/home/guiandreis/Airflow-tweets-spark-docker/airflow-spark-teste/data/silver")
from pyspark.sql.functions import col, to_date

# Filtra tweets de um dia específico, por exemplo 2026-02-10
df_filtered = df.filter(to_date(col("created_at")) == "2026-02-04")

df_filtered.show(20)





26/02/12 20:14:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+---------+--------------------+-----------------+--------+----------------+--------+--------------------+-----+------+-------+--------+---------------+------------+
|user_id|thread_id|          created_at|edit_versions_ids|tweet_id|reply_to_user_id|language|          tweet_text|likes|quotes|replies|retweets|processing_date|created_date|
+-------+---------+--------------------+-----------------+--------+----------------+--------+--------------------+-----+------+-------+--------+---------------+------------+
|     17|       36|2026-02-04T16:26:...|             [70]|      80|              90|      en|Tweet fictício ge...|   35|    70|     16|      34|     2026-02-12|  2026-02-04|
|     60|       34|2026-02-04T18:45:...|             [51]|      20|              55|      en|Outro tweet fictí...|   29|   100|     18|      69|     2026-02-12|  2026-02-04|
|      9|       86|2026-02-04T16:12:...|             [79]|      14|               2|      en|Este é um tweet f...|   87|    43|   

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, size,  count, avg
import pyspark.sql.functions as f 


spark = SparkSession.builder.appName("twitter_transformation_23").getOrCreate()

df_silver = spark.read.parquet('/home/guiandreis/Airflow-tweets-spark-docker/airflow-spark-teste/data/silver')


def add_metrics(df):
    """
    Adiciona a coluna 'engagement' somando likes, retweets, replies e quotes.
    """
    df = df.withColumn(
        "engagement",
        col("likes") + col("retweets") + col("replies") + col("quotes")
    )
    df= df.withColumn("num_edits", size(col("edit_versions_ids")))
    
    return df
    

def add_avg_eng_per_day(df):
    avg_eng = df.groupBy("created_date").agg(avg("engagement").alias("avg_eng_per_day"))
    return df.join(avg_eng, on="created_date", how="left")

def add_user_metrics(df):
    user_metrics = df.groupBy("user_id").agg(
        count("*").alias("total_tweets"),
        avg("engagement").alias("avg_eng_per_user")
    )
    return df.join(user_metrics, on="user_id", how="left")


def build_gold(df):
    df = add_metrics(df)
    df = add_avg_eng_per_day(df)
    df = add_user_metrics(df)
    return df.select(
        "user_id", "thread_id", "tweet_id", "created_at", "created_date",
    "tweet_text","likes", "quotes", "replies", "retweets", "engagement", 
    "avg_eng_per_day", "total_tweets", "avg_eng_per_user",
    "language")
df_silver = spark.read.parquet('/home/guiandreis/Airflow-tweets-spark-docker/airflow-spark-teste/data/silver')
gold_df = build_gold(df_silver)
gold_df.filter(col("created_date")=="2026-02-11").show(5, truncate=False)

# gold_df.show()


+-------+---------+--------+-------------------------------+------------+----------------------------------------------------------------------------------+-----+------+-------+--------+----------+------------------+------------+----------------+--------+
|user_id|thread_id|tweet_id|created_at                     |created_date|tweet_text                                                                        |likes|quotes|replies|retweets|engagement|avg_eng_per_day   |total_tweets|avg_eng_per_user|language|
+-------+---------+--------+-------------------------------+------------+----------------------------------------------------------------------------------+-----+------+-------+--------+----------+------------------+------------+----------------+--------+
|75     |22       |6       |2026-02-11T14:32:30.601694+0000|2026-02-11  |Tweet fictício criado usando inteligência artificial para falar sobre data science|87   |48    |93     |55      |283       |197.12857142857143|4           |201